In [1]:
import os
import torch
import clip
from PIL import Image
import numpy as np
import json

# Choose device: GPU if available, otherwise CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
# Load the CLIP model and preprocessing pipeline.
model, preprocess = clip.load("ViT-B/32", device=device)

# Specify the folder containing images.
image_folder = r"Dataset\cleaned_images"

# Create a mapping of image names to text descriptions
# You can customize this mapping based on your needs
# Load the mapping from JSON file

try:
    # Update the path to point to your JSON file
    with open('texts_cleaned.json', 'r') as f:
        image_to_text_mapping = json.load(f)
    print(f"Loaded {len(image_to_text_mapping)} image-text mappings from JSON file")
except FileNotFoundError:
    print("JSON mapping file not found. Using empty mapping.")
    image_to_text_mapping = {}
except json.JSONDecodeError:
    print("Error parsing JSON file. Using empty mapping.")
    image_to_text_mapping = {}

# Dictionary to store embeddings
image_embeddings_dict = {}
text_embeddings_dict = {}

# Iterate through all image files in the folder.
for image_file in os.listdir(image_folder):
    if image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):  # Check for valid image extensions.
        image_path = os.path.join(image_folder, image_file)
        try:
            # Extract the image name from the file name
            if image_file.startswith("interpolated_slip_image_") and image_file.endswith(".png"):
                image_name = image_file[24:-4]  # Remove prefix and ".png" suffix
                print(f"Processing image: {image_name}")
            else:
                image_name = os.path.splitext(image_file)[0]  # Just remove extension
                print(f"Processing file with unexpected format: {image_file}")
            
            # Load and preprocess the image.
            image = Image.open(image_path)
            image_input = preprocess(image).unsqueeze(0).to(device)

            with torch.no_grad():
                # Get image embedding
                image_embedding = model.encode_image(image_input)
                image_embeddings_dict[image_name] = image_embedding.cpu().numpy()
                
                # Get text embedding if available
                if image_name in image_to_text_mapping:
                    text = [image_to_text_mapping[image_name]]
                    text_input = clip.tokenize(text).to(device)
                    text_embedding = model.encode_text(text_input)
                    text_embeddings_dict[image_name] = text_embedding.cpu().numpy()
                    print(f"Generated embeddings for image and text: {image_name}")
                else:
                    print(f"No text description found for {image_name}")
                    
        except Exception as e:
            print(f"Error processing {image_file}: {e}")

    break

print(f"Processed {len(image_embeddings_dict)} images with embeddings")
print(f"Generated text embeddings for {len(text_embeddings_dict)} images")


# Optional: Save embeddings to disk
# np.save("image_embeddings.npy", image_embeddings_dict)
# np.save("text_embeddings.npy", text_embeddings_dict)

100%|███████████████████████████████████████| 338M/338M [01:17<00:00, 4.57MiB/s]


Loaded 73 image-text mappings from JSON file
Processing image: s1923KANTOJ01KOBA.fsp
Error processing interpolated_slip_image_s1923KANTOJ01KOBA.fsp.png: Input The value of the earthquake fault plane parameter "LEN_f" is 130.0.
The value of the earthquake fault plane parameter "WID" is 70.0.
The value of the earthquake fault plane parameter "Mw" is 8.08.
The value of the earthquake fault plane parameter "Mo" is 1.46e+21.
The value of the earthquake fault plane parameter "STRK" is 290.0.
The value of the earthquake fault plane parameter "DIP" is 25.0.
The value of the earthquake fault plane parameter "RAKE" is 140.0.
The value of the earthquake fault plane parameter "Htop" is 2.0.
The value of the earthquake fault plane parameter "HypX" is 110.5.
The value of the earthquake fault plane parameter "HypZ" is 35.0.
The value of the earthquake fault plane parameter "avTr" is 10.5.
The value of the earthquake fault plane parameter "avVr" is 2.6.
The value of the earthquake fault plane paramete

In [9]:
import os
import torch
from PIL import Image
import numpy as np
import json

# OpenFlamingo imports
from open_flamingo import create_model_and_transforms
from transformers import AutoTokenizer

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load OpenFlamingo
# model, image_processor, tokenizer = create_model_and_transforms(
#     model_name="OpenFlamingo-9B",
#     cache_dir="./checkpoints/OpenFlamingo-9B-vitl-mpt7b",
#     device=device
# )

model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",  # Not laion/CLIP-ViT-L-14
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path="mosaicml/mpt-7b",
    tokenizer_path="EleutherAI/gpt-neox-20b",  # Or whatever matches your lang encoder
    model_name="OpenFlamingo-9B",
    cache_dir="./checkpoints/OpenFlamingo-9B-vitl-mpt7b",
    device=device
)

model.eval()

# Paths
image_folder = r"Dataset\cleaned_images"

# Load image-to-text mapping
try:
    with open('texts_cleaned.json', 'r') as f:
        image_to_text_mapping = json.load(f)
    print(f"Loaded {len(image_to_text_mapping)} image-text mappings from JSON file")
except Exception as e:
    print(f"Error loading JSON file: {e}")
    image_to_text_mapping = {}

# Embedding storage
image_embeddings_dict = {}
text_embeddings_dict = {}

# Loop through files
for image_file in os.listdir(image_folder):
    if image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_path = os.path.join(image_folder, image_file)
        try:
            image_name = image_file[24:-4] if image_file.startswith("interpolated_slip_image_") else os.path.splitext(image_file)[0]
            print(f"Processing image: {image_name}")

            # Load image and preprocess
            image = Image.open(image_path).convert("RGB")
            vision_x = image_processor(image).unsqueeze(0).unsqueeze(1).to(device)  # [batch, num_images, C, H, W]

            # Get text prompt (must wrap in <image></image>)
            if image_name in image_to_text_mapping:
                prompt = f"<image>{image_to_text_mapping[image_name]}</image>"
                tokens = tokenizer(prompt, return_tensors="pt").to(device)

                # Generate image-text embedding (optional)
                with torch.no_grad():
                    outputs = model.generate(
                        vision_x=vision_x,
                        lang_x=tokens["input_ids"],
                        attention_mask=tokens["attention_mask"],
                        max_new_tokens=0,
                        return_dict_in_generate=True,
                        output_hidden_states=True
                    )

                    # You can use hidden states or final logits as embeddings
                    last_hidden = outputs.hidden_states[-1][0]  # shape: [seq_len, hidden_dim]
                    pooled = last_hidden.mean(dim=0).cpu().numpy()

                    image_embeddings_dict[image_name] = pooled
                    text_embeddings_dict[image_name] = pooled
                    print(f"Generated OpenFlamingo embeddings for {image_name}")
            else:
                print(f"No text found for {image_name}")
        except Exception as e:
            print(f"Error processing {image_file}: {e}")

    break  # Remove for full processing

print(f"Processed {len(image_embeddings_dict)} images with embeddings")
print(f"Generated text embeddings for {len(text_embeddings_dict)} images")


KeyboardInterrupt: 